# クラブヘッド検出モデルの学習（Lesson OS・Phase 1）

ゴルフクラブのヘッドとシャフトを画像から直接検出するモデル（YOLOX-Nano・2クラス）を学習し、
ブラウザで動かせる ONNX ファイルを作ります。

**手順（全部このノートブックの中で完結します）**
1. ランタイム → ランタイムのタイプを変更 → **GPU（T4）** を選ぶ
2. [Roboflow](https://roboflow.com) の無料アカウントを作り、Settings → API Key をコピーしておく
3. 上から順にセルを実行（▶）。学習は T4 で **1〜2時間** くらい
4. 最後のセルで `club_head_nano.onnx`（約4MB）がダウンロードされるので、
   Cowork のチャットに添付して「組み込んで」と言えば pose.ts への統合はこちらでやります

**ライセンスの約束（外販のため）**
- 学習フレームワーク: YOLOX = **Apache-2.0**（商用可）。Ultralytics YOLOv8/11 は AGPL なので使わない
- データセット: golf-club-tracking 11,479枚 = **CC BY 4.0**（アプリのクレジット表記に出典を書く）／club&head 300枚 = **CC0**


## 1. GPUの確認

In [ ]:
!nvidia-smi


## 2. データセットの取得（RoboflowのAPIキーが要ります）

In [ ]:
import getpass
ROBOFLOW_API_KEY = getpass.getpass("Roboflow API Key: ")


In [ ]:
%pip -q install roboflow
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# ① golf-club-tracking（11,479枚・CC BY 4.0）
p1 = rf.workspace("club-head-tracking").project("golf-club-tracking")
d1 = p1.version(2).download("coco", location="/content/rf_club_tracking")

# ② Golf club & head（300枚・CC0）— 最新バージョンを自動で選ぶ
p2 = rf.workspace("public-bezoe").project("golf-club---head-object-detection")
v2 = sorted(int(v.version.split("/")[-1]) for v in p2.versions())[-1]
d2 = p2.version(v2).download("coco", location="/content/rf_club_head")

print("done:", d1.location, d2.location)


## 3. 2つのデータセットを1つに束ねる

クラスを `club_head` / `club` の2つに寄せて、YOLOX標準のCOCOレイアウトに変換します。
実行ログに出る「クラス対応」で、`head` 系が 1、`club` 系が 2 になっていることを確認してください。
`None` になったクラス（ボールなど）は捨てられます。

In [ ]:
%%writefile /content/merge_coco.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Roboflow の COCO エクスポート（複数データセット）を1つに束ね、
YOLOX 標準の COCO レイアウト（train2017/val2017/annotations）に変換する。

クラスは2つに寄せる:
  1 = club_head （名前に head/Head を含むクラス）
  2 = club      （名前に club/shaft/grip を含むクラス。head を含むものは除く）
それ以外のクラス（ボール・人など）は捨てる。
"""
import json
import shutil
import sys
from pathlib import Path

OUT_CLASSES = [
    {"id": 1, "name": "club_head", "supercategory": "golf"},
    {"id": 2, "name": "club", "supercategory": "golf"},
]


def map_class(name: str):
    n = name.lower()
    if "head" in n:
        return 1
    if "club" in n or "shaft" in n or "grip" in n:
        return 2
    return None


def find_split_dir(ds: Path, names):
    for n in names:
        d = ds / n
        if (d / "_annotations.coco.json").exists():
            return d
    return None


def merge(dataset_dirs, out_dir):
    out = Path(out_dir)
    for split, out_img_dir, out_ann in [
        (["train"], out / "train2017", out / "annotations" / "instances_train2017.json"),
        (["valid", "val"], out / "val2017", out / "annotations" / "instances_val2017.json"),
    ]:
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_ann.parent.mkdir(parents=True, exist_ok=True)
        images, annotations = [], []
        img_id = 1
        ann_id = 1
        for di, ds in enumerate(dataset_dirs):
            d = find_split_dir(Path(ds), split)
            if d is None:
                print(f"  [skip] {ds} に {split} が無い")
                continue
            coco = json.loads((d / "_annotations.coco.json").read_text(encoding="utf-8"))
            catmap = {c["id"]: map_class(c["name"]) for c in coco["categories"]}
            kept_names = {c["name"]: map_class(c["name"]) for c in coco["categories"]}
            print(f"  {ds} {split}: クラス対応 {kept_names}")
            old2new_img = {}
            for im in coco["images"]:
                src = d / im["file_name"]
                if not src.exists():
                    continue
                new_name = f"ds{di}_{im['file_name']}"
                shutil.copyfile(src, out_img_dir / new_name)
                old2new_img[im["id"]] = img_id
                images.append({"id": img_id, "file_name": new_name,
                               "width": im["width"], "height": im["height"]})
                img_id += 1
            for an in coco["annotations"]:
                new_cat = catmap.get(an["category_id"])
                if new_cat is None or an["image_id"] not in old2new_img:
                    continue
                annotations.append({
                    "id": ann_id, "image_id": old2new_img[an["image_id"]],
                    "category_id": new_cat, "bbox": an["bbox"],
                    "area": an.get("area", an["bbox"][2] * an["bbox"][3]),
                    "iscrowd": an.get("iscrowd", 0), "segmentation": [],
                })
                ann_id += 1
        out_ann.write_text(json.dumps({
            "images": images, "annotations": annotations, "categories": OUT_CLASSES,
        }), encoding="utf-8")
        n_head = sum(1 for a in annotations if a["category_id"] == 1)
        n_club = sum(1 for a in annotations if a["category_id"] == 2)
        print(f"  -> {out_ann.name}: 画像{len(images)} / club_head {n_head} / club {n_club}")


if __name__ == "__main__":
    merge(sys.argv[1:-1], sys.argv[-1])


In [ ]:
!python /content/merge_coco.py /content/rf_club_tracking /content/rf_club_head /content/dataset


## 4. YOLOX（Apache-2.0）の準備と学習前チェック

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/Megvii-BaseDetection/YOLOX.git
%pip -q install -e ./YOLOX --no-deps
%pip -q install loguru pycocotools thop tabulate psutil tensorboard tqdm ninja
# 事前学習済みの重み（COCO・Apache-2.0）から転移学習する
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth -O /content/yolox_nano.pth
import yolox; print("YOLOX", yolox.__version__)


In [ ]:
%%writefile /content/club_head_exp.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# YOLOX-Nano ベースのクラブヘッド検出（club_head / club の2クラス）
# ライセンス: YOLOX は Apache-2.0（外販可）。Ultralytics系(AGPL)は使わないこと。
import os

import torch.nn as nn

from yolox.exp import Exp as MyExp


class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        self.depth = 0.33
        self.width = 0.25
        # ヘッドは小さい物体なので nano 既定の416ではなく640で学習する
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.random_size = (14, 22)
        self.mosaic_scale = (0.5, 1.5)
        self.mosaic_prob = 0.5
        self.enable_mixup = False

        self.num_classes = 2
        self.data_dir = os.environ.get("CLUB_DATA_DIR", "/content/dataset")
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"

        self.max_epoch = int(os.environ.get("CLUB_EPOCHS", "40"))
        self.no_aug_epochs = 5
        self.warmup_epochs = 2
        self.eval_interval = 5
        self.data_num_workers = 2
        self.exp_name = "club_head_nano"

    def get_model(self, sublinear=False):
        def init_yolo(M):
            for m in M.modules():
                if isinstance(m, nn.BatchNorm2d):
                    m.eps = 1e-3
                    m.momentum = 0.03

        if "model" not in self.__dict__:
            from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
            in_channels = [256, 512, 1024]
            backbone = YOLOPAFPN(
                self.depth, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            head = YOLOXHead(
                self.num_classes, self.width, in_channels=in_channels,
                act=self.act, depthwise=True,
            )
            self.model = YOLOX(backbone, head)

        self.model.apply(init_yolo)
        self.model.head.initialize_biases(1e-2)
        return self.model


## 5. 学習（T4で1〜2時間）

途中で止まっても `-c /content/YOLOX_outputs/club_head_nano/latest_ckpt.pth --resume` で再開できます。
最後に出る **AP(club_head)** をメモしてください（0.5以上あればまず使い物になります）。

In [ ]:
import os
os.environ["CLUB_DATA_DIR"] = "/content/dataset"
os.environ["CLUB_EPOCHS"] = "40"
%cd /content/YOLOX
!python tools/train.py -f /content/club_head_exp.py -d 1 -b 16 --fp16 -o -c /content/yolox_nano.pth


## 6. ONNXへ変換

公式の export_onnx.py は新しい PyTorch で動かない（私有APIを使っている）ので、
公開APIだけで書いた変換スクリプトを使います。decode込みで出すのでブラウザ側の実装が単純になります。

In [ ]:
%%writefile /content/export_club_onnx.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
YOLOX 学習済みチェックポイント → ONNX 変換（decode込み）。
公式 tools/export_onnx.py は torch.onnx._export（私有API・torch 2.13で削除）を
使っていて新しい torch で落ちるので、公開APIだけで書き直したもの。
出力: [1, N, 7] ではなく decode 済み [1, 8400, 5+num_classes]
      （cx, cy, w, h, obj, cls...。後段でしきい値＋NMSをかける）
"""
import argparse
import importlib.util

import torch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("-f", "--exp_file", required=True)
    ap.add_argument("-c", "--ckpt", required=True)
    ap.add_argument("-o", "--output", default="club_head_nano.onnx")
    args = ap.parse_args()

    spec = importlib.util.spec_from_file_location("exp_module", args.exp_file)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    exp = mod.Exp()

    model = exp.get_model()
    ckpt = torch.load(args.ckpt, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    model.eval()
    # decode をモデル内で済ませる（ブラウザ側のJSを単純にするため）
    model.head.decode_in_inference = True

    dummy = torch.randn(1, 3, exp.test_size[0], exp.test_size[1])
    torch.onnx.export(
        model, dummy, args.output,
        input_names=["images"], output_names=["output"],
        opset_version=17, dynamo=False,
    )
    print(f"saved {args.output} (input {tuple(dummy.shape)})")


if __name__ == "__main__":
    main()


In [ ]:
%cd /content
!python export_club_onnx.py -f club_head_exp.py -c /content/YOLOX_outputs/club_head_nano/best_ckpt.pth -o /content/club_head_nano.onnx
!ls -la /content/club_head_nano.onnx


## 7. 自分のスイング動画で試す

`_swing_sample` の IMG_8986.mov などをアップロードすると、コマごとに検出して
枠を描いた確認用動画を作ります（クラブヘッド=赤 / シャフト=青）。

In [ ]:
from google.colab import files
up = files.upload()  # ここで .mov / .mp4 をアップロード
video_path = "/content/" + list(up.keys())[0]


In [ ]:
import subprocess, glob, os
import numpy as np, cv2, torch
import onnxruntime as ort
from yolox.utils import postprocess

os.makedirs("/content/frames", exist_ok=True)
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", video_path,
                "-vf", "fps=30", "/content/frames/%04d.jpg"], check=True)
frames = sorted(glob.glob("/content/frames/*.jpg"))
print(len(frames), "frames")

sess = ort.InferenceSession("/content/club_head_nano.onnx", providers=["CPUExecutionProvider"])
SIZE = 640

def preproc(img):
    h, w = img.shape[:2]
    r = min(SIZE / h, SIZE / w)
    canvas = np.full((SIZE, SIZE, 3), 114, dtype=np.uint8)
    ri = cv2.resize(img, (int(w * r), int(h * r)))
    canvas[: ri.shape[0], : ri.shape[1]] = ri
    x = canvas.transpose(2, 0, 1)[None].astype(np.float32)
    return x, r

os.makedirs("/content/out_frames", exist_ok=True)
hits = 0
for i, f in enumerate(frames):
    img = cv2.imread(f)
    x, r = preproc(img)
    out = sess.run(None, {"images": x})[0]
    dets = postprocess(torch.from_numpy(out), num_classes=2, conf_thre=0.3, nms_thre=0.45)[0]
    if dets is not None:
        hits += 1
        for d in dets.numpy():
            x1, y1, x2, y2 = (d[:4] / r).astype(int)
            cls = int(d[6]); conf = d[4] * d[5]
            color = (0, 0, 255) if cls == 0 else (255, 128, 0)  # club_head=赤 / club=青
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, f"{'head' if cls==0 else 'club'} {conf:.2f}", (x1, max(0, y1-6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    cv2.imwrite(f"/content/out_frames/{i:04d}.jpg", img)

print(f"検出できたコマ: {hits}/{len(frames)}")
subprocess.run(["ffmpeg", "-y", "-v", "error", "-framerate", "30",
                "-i", "/content/out_frames/%04d.jpg", "-c:v", "libx264",
                "-pix_fmt", "yuv420p", "/content/check.mp4"], check=True)
print("確認用動画: /content/check.mp4")


In [ ]:
# 確認用動画をその場で見る
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("/content/check.mp4", "rb").read()).decode()
HTML(f'<video width=360 controls src="data:video/mp4;base64,{data}"></video>')


## 8. 成果物のダウンロード

`club_head_nano.onnx` を Cowork のチャットに添付してください。pose.ts への統合
（onnxruntime-web・差分方式との融合・既存DPでの絞り込み）はこちらでやります。
`check.mp4` も一緒にもらえると精度の判断が早いです。

In [ ]:
from google.colab import files
files.download("/content/club_head_nano.onnx")
files.download("/content/check.mp4")
